<a href="https://colab.research.google.com/github/kuruvajayanth12/LargeLanguageModel-LLM-/blob/main/Single_Multi_Head_Attention'.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

def softmax(x):
    # subtract max for numerical stability
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def single_head_attention(X, W_q, W_k, W_v):
    """
    X   : (seq_len, d_model)
    W_q : (d_model, d_k)
    W_k : (d_model, d_k)
    W_v : (d_model, d_v)
    """

    # 1. Linear projections
    Q = X @ W_q   # (seq_len, d_k)
    K = X @ W_k   # (seq_len, d_k)
    V = X @ W_v   # (seq_len, d_v)

    # 2. Scaled dot-product attention
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)   # (seq_len, seq_len)

    # 3. Softmax over keys
    weights = softmax(scores)         # (seq_len, seq_len)

    # 4. Weighted sum of values
    output = weights @ V              # (seq_len, d_v)

    return output, weights


In [2]:
np.random.seed(0)

# Example input: 3 tokens, embedding size 4
X = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 2.0],
    [1.0, 1.0, 1.0, 1.0]
])

d_model = 4
d_k = 2
d_v = 2

# Random weight matrices
W_q = np.random.randn(d_model, d_k)
W_k = np.random.randn(d_model, d_k)
W_v = np.random.randn(d_model, d_v)

output, attention_weights = single_head_attention(X, W_q, W_k, W_v)

print("Attention weights:")
print(attention_weights)

print("\nOutput:")
print(output)


Attention weights:
[[2.33986881e-01 2.55541065e-01 5.10472055e-01]
 [2.93496318e-05 9.71219545e-01 2.87511050e-02]
 [3.73204778e-03 7.41436164e-01 2.54831788e-01]]

Output:
[[ 0.41456847 -1.29680656]
 [ 2.29060821 -3.13362667]
 [ 1.77235735 -2.65787327]]


In [3]:
import numpy as np

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def multi_head_attention(X, W_q, W_k, W_v, W_o, num_heads):
    """
    X      : (seq_len, d_model)
    W_q,k,v: (d_model, d_model)
    W_o    : (d_model, d_model)
    """

    seq_len, d_model = X.shape
    d_k = d_model // num_heads

    # 1. Linear projections
    Q = X @ W_q  # (seq_len, d_model)
    K = X @ W_k
    V = X @ W_v

    # 2. Split into heads
    Q = Q.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    K = K.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    V = V.reshape(seq_len, num_heads, d_k).transpose(1, 0, 2)
    # shape now: (num_heads, seq_len, d_k)

    # 3. Scaled dot-product attention (per head)
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)
    weights = softmax(scores)  # (num_heads, seq_len, seq_len)
    head_outputs = weights @ V  # (num_heads, seq_len, d_k)

    # 4. Concatenate heads
    concat = head_outputs.transpose(1, 0, 2).reshape(seq_len, d_model)

    # 5. Final linear projection
    output = concat @ W_o

    return output, weights


In [4]:
np.random.seed(0)

# Input: 3 tokens, model dimension 4
X = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 2.0],
    [1.0, 1.0, 1.0, 1.0]
])

d_model = 4
num_heads = 2

# Weight matrices
W_q = np.random.randn(d_model, d_model)
W_k = np.random.randn(d_model, d_model)
W_v = np.random.randn(d_model, d_model)
W_o = np.random.randn(d_model, d_model)

output, attention_weights = multi_head_attention(
    X, W_q, W_k, W_v, W_o, num_heads
)

print("Output:")
print(output)

print("\nAttention weights shape:")
print(attention_weights.shape)  # (num_heads, seq_len, seq_len)


Output:
[[ 2.94779514  2.40228494  2.50425278 -2.75567962]
 [ 3.7801259   2.9612929   2.89817283 -3.20320388]
 [ 3.87723439  3.01727773  3.02840854 -2.80079157]]

Attention weights shape:
(2, 3, 3)
